In [ ]:
from __future__ import annotations
from os import PathLike

from fastcore.basics import patch


from trouver.obsidian.vault import VaultNote
from trouver.obsidian.file import MarkdownFile

In [ ]:
import os
from pathlib import Path 
import tempfile

from fastcore.test import *
from nbdev.showdoc import show_doc

from trouver.helper.tests import make_example_vault

In [ ]:
#| export obsidian.file
@patch
def write(
    self: MarkdownFile,
    vn: VaultNote, # Represents the file.
    mode: str = 'w', # The specific mode to write the file with.
    # enquote_entries_in_metadata_fields: list[str] = [] # A list of str of fields in the YAML metadata whose entries need to be enquoted. If there is a string that is not a key of `new_metadata`, then that string is essentially ignored (in particular, no errors are raised).
    check_validity: bool = True # If `True`, then check that the frontmatter YAML metadata parses (via `.metadata`) before writing to the file. If not, then a `ValueError` is raised.
    ) -> None:
    r"""
    Write to the file specified by a `VaultNote` object.

    If the file that the `VaultNote` object represents does not exist,
    then this method creates it.

    **Raises**
    - `ValueError`
        - If `check_validity` is `True` and if the frontmatter YAML metadata
        cannot be parsed by `pyyaml`. The metadata field elements then need to
        be escaped and/or enquoted.    
        It is recommended to use `.replace_metadata` and passing arguments to the
        `enquote_entries_in_fields` parameter or to use the `dict_to_metadata_lines`
        function to ensure that metadata fields are
        enquoted and escaped.
    """
    if not vn.exists():
        vn.create()
    # if enquote_entries_in_metadata_fields:
    #     self.replace_metadata(
    #         self.metadata(), enquote_entries_in_metadata_fields)
    if check_validity:
        try:
            self.metadata()
        except Exception as e:
            raise ValueError(fr"""Tried to write the MarkdownFile object to {vn.path()}, 
but its frontmatter YAML metadata cannot be parsed by pyyaml.
Most likely, the YAML frontmatter cannot be parsed, and metadata field elements need to
be enquoted and/or escaped (say via using double slashes `\\` instead of single slashes `\`
and by using `\"` instead of just `"`). It is recommended to apply the `.replace_metadata` method of the
`MarkdownFile` object, specifying arguments for the `enquote_entries_in_fields` parameter.
The following is the MarkdownFile object's str representation:
\n\n{str(self)}
""")
    vn.write(str(self), mode, 'utf-8')

    # with open(vn.path(), mode, encoding='utf-8') as file:
    #     file.write(str(self))
    #     file.close()


In [ ]:
show_doc(MarkdownFile.write)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/file.py#L1573){target="_blank" style="float:right; font-size:smaller"}

### MarkdownFile.write

>      MarkdownFile.write (vn:trouver.obsidian.vault.VaultNote, mode:str='w',
>                          check_validity:bool=True)

*Write to the file specified by a `VaultNote` object.

If the file that the `VaultNote` object represents does not exist,
then this method creates it.

**Raises**
- `ValueError`
    - If `check_validity` is `True` and if `str(self)` cannot be parsed
    as a `MarkdownFile` object. If this error is raised, then most likely the
    issue is that the frontmatter metadata cannot be parsed by `pyyaml` and
    the metadata field elements need to be escaped and/or enquoted.    
    It is recommended to use `.replace_metadata` and passing arguments to the
    `enquote_entries_in_fields` parameter or to use the `dict_to_metadata_lines`
    function to ensure that metadata fields are
    enquoted and escaped.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| vn | VaultNote |  | Represents the file. |
| mode | str | w | The specific mode to write the file with. |
| check_validity | bool | True | If `True`, then check whether a MarkdownFile object could be instantiated from `self.__str__()` before writing to the file. If not, then a `ValueError` is raised. |
| **Returns** | **None** |  |  |

The `MarkdownFile.write` method raises a `ValueError` if `check_validity` is `True` and if the frontmatter YAML metadata of the `MarkdownFile` object cannot be parsed by `pyyaml`.


In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir_', dir=os.getcwd()) as temp_dir:
    make_example_vault(temp_dir)

    # The following text is parseable by pyyaml.
    metadata_markdownfile_text = r'''---
latex_in_original: ["\\text{hi}"]
---
blahblah
'''
    mf = MarkdownFile.from_string(metadata_markdownfile_text)
    print("Note that `mf` is completely fine right after it is constructed:")
    print(str(mf))
    MarkdownFile.from_string(str(mf))  # This does not raise errors

    mf.add_tags('_new_tag')
    print('\nAfter adding a tag, however, the metadata changes. Note that the `latex_in_original` field is no longer enquoted and escaped:')
    print(mf.__str__())
    print('\n\nTo prevent further problems, the `.write` method raises a `ValueError`')

    vault_note = VaultNote(temp_dir, name='pauli_wickelgren_aa1_example 3.7')

    with ExceptionExpected(ex=ValueError):
        mf.write(vault_note)


Note that `mf` is completely fine right after it is constructed:
---
latex_in_original: ["\\text{hi}"]
---
blahblah

After adding a tag, however, the metadata changes. Note that the `latex_in_original` field is no longer enquoted and escaped:
---
latex_in_original: [\text{hi}]
tags: [_new_tag]
---
blahblah


To prevent further problems, the `.write` method raises a `ValueError`


If `check_validity` is `False`, then the `.write` method will attempt to write to the file of the `VaultNote` object. However, note that the frontmatter metadata of the resulting file may not be parseable by `pyyaml`.


In [ ]:

with tempfile.TemporaryDirectory(prefix='temp_dir_', dir=os.getcwd()) as temp_dir:
    make_example_vault(temp_dir)

    # The following text is parseable by pyyaml.
    metadata_markdownfile_text = r'''---
latex_in_original: ["\\text{hi}"]
---
blahblah
'''
    mf = MarkdownFile.from_string(metadata_markdownfile_text)
    print("Note that `mf` is completely fine right after it is constructed:")
    print(str(mf))
    MarkdownFile.from_string(str(mf))  # This does not raise errors

    mf.add_tags('_new_tag')
    print('\nAfter adding a tag, however, the metadata changes. Note that the `latex_in_original` field is no longer enquoted and escaped:')
    print(mf.__str__())
    print('\n\nTo prevent further problems, the `.write` method raises a `ValueError`')

    vault_note = VaultNote(temp_dir, name='pauli_wickelgren_aa1_example 3.7')

    mf.write(vault_note, check_validity=False)
    new_mf = MarkdownFile.from_vault_note(vault_note)
    with ExceptionExpected(ValueError): # The ValueError is from the .metadata method.
        new_mf.metadata()


Note that `mf` is completely fine right after it is constructed:
---
latex_in_original: ["\\text{hi}"]
---
blahblah

After adding a tag, however, the metadata changes. Note that the `latex_in_original` field is no longer enquoted and escaped:
---
latex_in_original: [\text{hi}]
tags: [_new_tag]
---
blahblah


To prevent further problems, the `.write` method raises a `ValueError`


In [ ]:
# P3: the validity check catches unparseable frontmatter (yaml only — no full re-parse)
with tempfile.TemporaryDirectory(prefix='temp_dir_', dir=os.getcwd()) as temp_dir:
    make_example_vault(temp_dir)
    mf = MarkdownFile.from_string(r'''---
latex_in_original: ["\\text{hi}"]
---
blahblah
''')
    mf.add_tags('_new_tag')  # the metadata round-trip drops the escaping → unparseable yaml
    vault_note = VaultNote(temp_dir, name='pauli_wickelgren_aa1_example 3.7')
    with ExceptionExpected(ValueError):
        mf.write(vault_note)
    # valid frontmatter writes fine
    mf2 = MarkdownFile.from_string('---\ntags: [a]\n---\nbody')
    vault_note2 = VaultNote(temp_dir, name='pauli_wickelgren_aa1_example 3.7')
    mf2.write(vault_note2)
    assert vault_note2.exists()
